# Portfolio Risk & Robo-Advisor Allocation Engine

**Data:** real daily price data for 30 tickers (2013–2018), `../data/raw/stock_prices.csv`; synthetic client profiles, `../data/raw/client_profiles.csv` (see README.md "Data Note").

SQLite DB (from ETL): `../data/processed/portfolio.db`. Exploratory SQL: `../sql/queries.sql`.

## 1. Load & Inspect Data

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PRICES_CSV = Path("../data/raw/stock_prices.csv")
PROFILES_CSV = Path("../data/raw/client_profiles.csv")
DB_PATH = Path("../data/processed/portfolio.db")
QUERIES_PATH = Path("../sql/queries.sql")

prices = pd.read_csv(PRICES_CSV, parse_dates=["date"])
profiles = pd.read_csv(PROFILES_CSV)
prices.head()

In [ ]:
prices_wide = (
    prices.sort_values("date")
    .pivot(index="date", columns="Name", values="close")
    .sort_index()
)
prices_wide.head()

## 2. SQL Sanity Checks & Returns

Run the exploratory queries in `sql/queries.sql` against `data/processed/portfolio.db`, then compute pandas returns and spot-check AAPL against the SQL `LAG()` daily change.

In [ ]:
def run_query(sql: str, db_path: Path = DB_PATH) -> pd.DataFrame:
    """Execute a SQL string against the portfolio SQLite DB and return a DataFrame."""
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(sql, conn)


print(f"DB exists: {DB_PATH.resolve().exists()}")
print(f"Queries file: {QUERIES_PATH.resolve()}")

### 2.1 Date range / ticker count sanity check

In [ ]:
q1 = """
SELECT
    MIN(date) AS start_date,
    MAX(date) AS end_date,
    COUNT(DISTINCT ticker) AS n_tickers,
    COUNT(*) AS n_rows
FROM prices
"""
print(q1.strip())
run_query(q1)

### 2.2 Average close price by ticker

In [ ]:
q2 = """
SELECT
    ticker,
    ROUND(AVG(close), 2) AS avg_close
FROM prices
GROUP BY ticker
ORDER BY avg_close DESC
"""
print(q2.strip())
run_query(q2)

### 2.3 Client counts and average age by risk tier

In [ ]:
q3 = """
SELECT
    risk_tier,
    COUNT(*) AS num_clients,
    ROUND(AVG(age), 1) AS avg_age
FROM client_profiles
GROUP BY risk_tier
ORDER BY risk_tier
"""
print(q3.strip())
run_query(q3)

### 2.4 Cross-tab of risk tier × income bracket

In [ ]:
q4 = """
SELECT
    risk_tier,
    income_bracket,
    COUNT(*) AS num_clients
FROM client_profiles
GROUP BY risk_tier, income_bracket
ORDER BY risk_tier, income_bracket
"""
print(q4.strip())
run_query(q4)

### 2.5 AAPL daily change via `LAG()` (SQL-side check vs pandas)

In [ ]:
q5 = """
SELECT
    date,
    ticker,
    close,
    LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS prev_close,
    close - LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS daily_change,
    (close - LAG(close) OVER (PARTITION BY ticker ORDER BY date))
        / LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS daily_return
FROM prices
WHERE ticker = 'AAPL'
ORDER BY date
"""
print(q5.strip())
aapl_sql = run_query(q5)
aapl_sql["date"] = pd.to_datetime(aapl_sql["date"])
aapl_sql.head(10)

### 2.6 Pandas returns + AAPL spot-check (SQL `LAG` vs pandas)

Compute wide daily percentage returns in pandas, then compare SQL dollar change / return against pandas `diff` / `pct_change` on several dates.

In [ ]:
# Daily percentage returns (drop first all-NaN row from pct_change)
returns = prices_wide.pct_change().dropna(how="all")
print(f"Returns shape: {returns.shape[0]:,} days × {returns.shape[1]} tickers")
returns.head()

In [ ]:
# Pandas AAPL series aligned for comparison
aapl_px = prices_wide["AAPL"].dropna().sort_index()
aapl_pandas = pd.DataFrame(
    {
        "close": aapl_px,
        "pandas_daily_change": aapl_px.diff(),
        "pandas_daily_return": aapl_px.pct_change(),
    }
).reset_index()

spot_dates = [
    "2013-02-11",
    "2014-06-09",
    "2015-08-24",
    "2016-01-15",
    "2017-12-29",
]

comparison = (
    aapl_sql.merge(aapl_pandas, on="date", how="inner", suffixes=("_sql", "_px"))
    .loc[lambda d: d["date"].isin(pd.to_datetime(spot_dates))]
    .assign(
        change_match=lambda d: np.isclose(
            d["daily_change"], d["pandas_daily_change"], rtol=0, atol=1e-9, equal_nan=True
        ),
        return_match=lambda d: np.isclose(
            d["daily_return"], d["pandas_daily_return"], rtol=0, atol=1e-12, equal_nan=True
        ),
    )[
        [
            "date",
            "close_sql",
            "daily_change",
            "pandas_daily_change",
            "daily_return",
            "pandas_daily_return",
            "change_match",
            "return_match",
        ]
    ]
)

print("AAPL spot-check: SQL LAG vs pandas (side by side)")
display(comparison)
assert comparison["change_match"].all() and comparison["return_match"].all(), (
    "SQL LAG and pandas returns disagree on spot-checked dates"
)
print("All spot-checked dates match.")

### 2.7 Covariance (placeholder for PyPortfolioOpt)

Expected returns / sample covariance will be filled in with PyPortfolioOpt in a later step.

In [ ]:
# TODO: from pypfopt import expected_returns, risk_models
# mu = expected_returns.mean_historical_return(prices_wide)
# S = risk_models.sample_cov(prices_wide)

## 3. Efficient Frontier & Risk-Tier Portfolios

Build a max-Sharpe portfolio at each of three target volatility levels (Conservative / Moderate / Aggressive) using PyPortfolioOpt.

In [ ]:
# TODO: from pypfopt import EfficientFrontier
# ef = EfficientFrontier(mu, S)
# weights = ef.efficient_risk(target_volatility=...)

## 4. Risk Metrics: VaR & Sharpe Ratio by Tier

In [ ]:
# TODO: historical VaR at 95% confidence, annualized Sharpe ratio, per tier

## 5. Rebalancing Simulation

Simulate a rule that rebalances when any position drifts more than 5% from its target weight. Count how often this would have triggered historically.

In [ ]:
# TODO: rolling weight drift simulation

## 6. Benchmark Comparison: 60/40 Portfolio

Compare risk-adjusted return (Sharpe ratio) of each risk tier against a standard 60/40 (equity/bond-proxy) benchmark over the same period.

In [ ]:
# TODO: construct 60/40 benchmark, compare Sharpe ratios

## 7. Client Assignment

Map each synthetic client profile to a risk-tier portfolio based on `risk_tolerance_score`, and summarize expected outcomes.

In [ ]:
# TODO: merge profiles with tier portfolio performance